In [ ]:
# Model Evaluation and Explainability for Loan Default Risk Assessment
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import joblib
import pickle
import os
import shap
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, precision_recall_curve, auc,
    confusion_matrix, classification_report, log_loss,
    roc_curve, average_precision_score
)

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('ggplot')
sns.set(style='whitegrid')

# --------- 1. Load the Model ---------
# Update this path to where your model is saved
MODEL_PATH = '../model/xgboost_loan_default_model.pkl'

try:
    # Try different loading methods depending on how model was saved
    try:
        model = pickle.load(open(MODEL_PATH, 'rb'))
        print("Model loaded with pickle")
    except:
        try:
            model = joblib.load(MODEL_PATH)
            print("Model loaded with joblib")
        except:
            try:
                model = xgb.Booster()
                model.load_model(MODEL_PATH)
                print("Model loaded with XGBoost native format")
            except:
                raise Exception(f"Couldn't load model from {MODEL_PATH}")
    
    print(f"Successfully loaded model from {MODEL_PATH}")
    # Print model parameters if available
    if hasattr(model, 'get_params'):
        print("Model parameters:")
        for param, value in model.get_params().items():
            print(f"  {param}: {value}")
except Exception as e:
    print(f"Error loading model: {e}")

# --------- 2. Load Evaluation Data ---------
# Update this path to your evaluation dataset
EVAL_DATA_PATH = '/mnt/object/loan-default-data/eval_data.csv'

try:
    eval_data = pd.read_csv(EVAL_DATA_PATH)
    print(f"Evaluation data loaded: {eval_data.shape[0]} rows, {eval_data.shape[1]} columns")
    
    # Display first few rows and data info
    print("First 5 rows of evaluation data:")
    display(eval_data.head())
    
    print("\nData info:")
    display(eval_data.info())
    
    print("\nSummary statistics:")
    display(eval_data.describe())
    
    # Check for missing values
    missing_values = eval_data.isnull().sum()
    if missing_values.sum() > 0:
        print("\nMissing values in evaluation data:")
        display(missing_values[missing_values > 0])
    else:
        print("\nNo missing values in evaluation data.")
    
    # Check class distribution
    target_col = 'loan_default'  # Update to your target column name
    class_dist = eval_data[target_col].value_counts(normalize=True)
    print("\nClass distribution in evaluation data:")
    display(class_dist)
    
    # Create X and y for evaluation
    X_eval = eval_data.drop(columns=[target_col])
    y_eval = eval_data[target_col]
    
except Exception as e:
    print(f"Error loading evaluation data: {e}")

# --------- 3. Make Predictions ---------
try:
    # Check if model is sklearn/xgboost API compatible or Booster
    if hasattr(model, 'predict_proba'):
        # For sklearn API
        y_pred_proba = model.predict_proba(X_eval)[:, 1]
        y_pred = model.predict(X_eval)
    else:
        # For native XGBoost Booster
        deval = xgb.DMatrix(X_eval)
        y_pred_proba = model.predict(deval)
        y_pred = (y_pred_proba > 0.5).astype(int)  # Using 0.5 as threshold
    
    print(f"Made predictions on {len(y_pred)} samples")
    print(f"Prediction probabilities range: {y_pred_proba.min():.4f} to {y_pred_proba.max():.4f}")
    
    # Display a preview of predictions
    pred_df = pd.DataFrame({
        'Actual': y_eval,
        'Predicted': y_pred,
        'Probability': y_pred_proba
    }).head(10)
    display(pred_df)
    
except Exception as e:
    print(f"Error making predictions: {e}")

# --------- 4. Calculate Classification Metrics ---------
try:
    # Basic metrics
    accuracy = accuracy_score(y_eval, y_pred)
    precision = precision_score(y_eval, y_pred)
    recall = recall_score(y_eval, y_pred)
    f1 = f1_score(y_eval, y_pred)
    roc_auc = roc_auc_score(y_eval, y_pred_proba)
    avg_precision = average_precision_score(y_eval, y_pred_proba)
    logloss = log_loss(y_eval, y_pred_proba)
    
    # Create metrics table
    metrics_dict = {
        'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC AUC', 'PR AUC', 'Log Loss'],
        'Value': [accuracy, precision, recall, f1, roc_auc, avg_precision, logloss]
    }
    metrics_df = pd.DataFrame(metrics_dict)
    display(metrics_df)
    
    # Confusion Matrix
    plt.figure(figsize=(10, 8))
    cm = confusion_matrix(y_eval, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['No Default', 'Default'],
                yticklabels=['No Default', 'Default'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()
    
    # Classification Report
    print("\nClassification Report:")
    print(classification_report(y_eval, y_pred))
    
    # ROC Curve
    plt.figure(figsize=(10, 8))
    fpr, tpr, _ = roc_curve(y_eval, y_pred_proba)
    plt.plot(fpr, tpr, lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()
    
    # Precision-Recall Curve
    plt.figure(figsize=(10, 8))
    precision_curve, recall_curve, _ = precision_recall_curve(y_eval, y_pred_proba)
    plt.plot(recall_curve, precision_curve, lw=2, 
             label=f'PR curve (AP = {avg_precision:.3f})')
    plt.axhline(y=class_dist[1], color='r', linestyle='--', 
                label=f'Baseline (ratio = {class_dist[1]:.3f})')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend(loc="lower left")
    plt.grid(True)
    plt.show()
    
    # Threshold Analysis
    thresholds = np.linspace(0.1, 0.9, 9)
    threshold_metrics = []
    
    for threshold in thresholds:
        y_pred_t = (y_pred_proba >= threshold).astype(int)
        threshold_metrics.append({
            'Threshold': threshold,
            'Accuracy': accuracy_score(y_eval, y_pred_t),
            'Precision': precision_score(y_eval, y_pred_t),
            'Recall': recall_score(y_eval, y_pred_t),
            'F1-Score': f1_score(y_eval, y_pred_t)
        })
    
    threshold_df = pd.DataFrame(threshold_metrics)
    display(threshold_df)
    
    # Plot threshold metrics
    plt.figure(figsize=(12, 8))
    for column in threshold_df.columns:
        if column != 'Threshold':
            plt.plot(threshold_df['Threshold'], threshold_df[column], 
                     marker='o', label=column)
    
    plt.xlabel('Threshold')
    plt.ylabel('Score')
    plt.title('Impact of Classification Threshold on Metrics')
    plt.legend()
    plt.grid(True)
    plt.show()
    
except Exception as e:
    print(f"Error calculating classification metrics: {e}")

# --------- 5. SHAP Explanations ---------
try:
    print("Generating SHAP explanations...")
    
    # Prepare the SHAP explainer
    if hasattr(model, 'predict_proba'):  # sklearn API
        explainer = shap.Explainer(model)
        shap_values = explainer(X_eval)
    else:  # Native Booster
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_eval)
    
    # SHAP Summary Plot - Feature Importance Overview
    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_values, X_eval, plot_type="bar", show=False)
    plt.title("SHAP Feature Importance")
    plt.tight_layout()
    plt.show()
    
    # SHAP Summary Plot - Impact on Model Output
    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_values, X_eval, show=False)
    plt.title("SHAP Feature Impact Summary")
    plt.tight_layout()
    plt.show()
    
    # SHAP Dependence Plots for top features
    if hasattr(model, 'feature_importances_'):
        # For sklearn API
        importances = model.feature_importances_
        feature_names = X_eval.columns
        indices = np.argsort(importances)[-5:]  # Top 5 features
    else:
        # Try to get feature importance from SHAP values
        importances = np.abs(shap_values).mean(0)
        feature_names = X_eval.columns
        indices = np.argsort(importances)[-5:]
    
    # Plot dependence plots for top features
    for i in indices:
        plt.figure(figsize=(12, 8))
        feature_name = feature_names[i]
        shap.dependence_plot(feature_name, shap_values, X_eval, show=False)
        plt.title(f"SHAP Dependence Plot for {feature_name}")
        plt.tight_layout()
        plt.show()
    
    # SHAP Force Plot for individual explanations
    # Sample 3 predicted defaults and 3 non-defaults
    default_indices = np.where((y_pred == 1) & (y_eval == 1))[0][:3]
    non_default_indices = np.where((y_pred == 0) & (y_eval == 0))[0][:3]
    
    for idx in np.concatenate([default_indices, non_default_indices]):
        print(f"\nExplanation for sample {idx}:")
        print(f"  Actual: {'Default' if y_eval.iloc[idx] == 1 else 'No Default'}")
        print(f"  Predicted: {'Default' if y_pred[idx] == 1 else 'No Default'}")
        print(f"  Probability of Default: {y_pred_proba[idx]:.4f}")
        
        # Display the sample data
        print("  Sample features:")
        sample = X_eval.iloc[idx:idx+1]
        display(sample.T)
        
        # Generate force plot for this sample
        plt.figure(figsize=(20, 3))
        if hasattr(model, 'predict_proba'):
            shap.plots.waterfall(shap_values[idx], max_display=10, show=False)
        else:
            shap.force_plot(
                explainer.expected_value,
                shap_values[idx],
                X_eval.iloc[idx],
                feature_names=X_eval.columns,
                matplotlib=True,
                show=False
            )
        plt.title(f"SHAP Force Plot - Sample {idx}")
        plt.tight_layout()
        plt.show()
    
    # Decision plot for the same samples
    plt.figure(figsize=(12, 10))
    if hasattr(model, 'predict_proba'):
        shap.decision_plot(
            explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
            shap_values[np.concatenate([default_indices, non_default_indices])],
            X_eval.iloc[np.concatenate([default_indices, non_default_indices])],
            show=False
        )
    else:
        shap.decision_plot(
            explainer.expected_value if isinstance(explainer.expected_value, float) else explainer.expected_value[0],
            shap_values[np.concatenate([default_indices, non_default_indices])],
            X_eval.iloc[np.concatenate([default_indices, non_default_indices])],
            feature_names=X_eval.columns,
            show=False
        )
    plt.title("SHAP Decision Plot - Compare Samples")
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Error generating SHAP explanations: {e}")

# --------- 6. Analyze Errors ---------
try:
    # Find the most significant misclassifications
    error_df = pd.DataFrame({
        'Actual': y_eval,
        'Predicted': y_pred,
        'Probability': y_pred_proba,
        'Error': y_eval != y_pred,
        'Confidence': np.where(y_pred == 1, y_pred_proba, 1 - y_pred_proba)
    })
    
    # Add the original features
    error_df = pd.concat([error_df, X_eval.reset_index(drop=True)], axis=1)
    
    # Get the misclassifications
    misclassified = error_df[error_df['Error']]
    
    # Sort by confidence (most confident mistakes first)
    misclassified_sorted = misclassified.sort_values('Confidence', ascending=False)
    
    print("\nTop 10 Most Confident Misclassifications:")
    display(misclassified_sorted.head(10))
    
    # Analyze misclassification patterns
    print("\nMisclassification Analysis:")
    print(f"  Total misclassifications: {len(misclassified)}")
    print(f"  False Positives: {len(misclassified[misclassified['Predicted'] > misclassified['Actual']])}")
    print(f"  False Negatives: {len(misclassified[misclassified['Predicted'] < misclassified['Actual']])}")
    
    # Calculate error rates for different segments
    print("\nError Rates by Feature Segments:")
    for column in X_eval.select_dtypes(include=['number']).columns[:5]:  # Limit to first 5 numeric features
        try:
            # Create 4 bins for the feature
            error_df[f'{column}_bin'] = pd.qcut(error_df[column], 4, duplicates='drop')
            segment_error = error_df.groupby(f'{column}_bin')['Error'].mean()
            print(f"\n  {column} segment error rates:")
            display(segment_error)
            
            # Plot error rate by bin
            plt.figure(figsize=(10, 6))
            segment_error.plot(kind='bar')
            plt.title(f'Error Rate by {column} Segments')
            plt.ylabel('Error Rate')
            plt.xlabel(column)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
        except:
            print(f"  Could not segment by {column}")
    
except Exception as e:
    print(f"Error analyzing misclassifications: {e}")

# --------- 7. Save Results and Conclusions ---------
try:
    # Summarize the evaluation
    print("\n===== Model Evaluation Summary =====")
    print(f"Model: XGBoost Classifier")
    print(f"Evaluation Dataset: {EVAL_DATA_PATH}")
    print(f"Dataset Size: {len(y_eval)} samples")
    print(f"Class Distribution: {class_dist.to_dict()}")
    print("\nPerformance Metrics:")
    for i, row in metrics_df.iterrows():
        print(f"  {row['Metric']}: {row['Value']:.4f}")
    
    print("\nKey Findings:")
    print("  1. [Replace with your observation about model performance]")
    print("  2. [Replace with your observation about important features]")
    print("  3. [Replace with your observation about error patterns]")
    print("  4. [Replace with your observation about potential improvements]")
    
    # Save results to disk
    results_dir = '../eval_results'
    os.makedirs(results_dir, exist_ok=True)
    
    # Save metrics
    metrics_df.to_csv(f"{results_dir}/classification_metrics.csv", index=False)
    
    print(f"\nResults saved to {results_dir}")
    
except Exception as e:
    print(f"Error saving results: {e}")